# 03 — Feature Engineering

This notebook transforms the preprocessed hourly meteorological dataset into features suitable for future Machine Learning experiments.

The feature engineering stage includes:

- temporal features;
- cyclical time representations;
- lagged meteorological variables;
- rolling precipitation aggregates;
- short-term meteorological changes;
- future precipitation values from +1h to +6h.

Future precipitation columns are created only as target candidates and must never be used as model input features.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. FILE PATHS
# ============================================================

INPUT_PATH = Path("../data/processed/recife_2018_preprocessed.csv")
OUTPUT_PATH = Path("../data/processed/recife_2018_features.csv")

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {INPUT_PATH}")


# ============================================================
# 2. LOAD PREPROCESSED DATASET
# ============================================================

df = pd.read_csv(
    INPUT_PATH,
    parse_dates=["datetime_utc"]
)

df = (
    df
    .sort_values("datetime_utc")
    .set_index("datetime_utc")
)

print("Dataset loaded:", df.shape)


# ============================================================
# 3. VALIDATE HOURLY TIMELINE
# ============================================================

expected_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="h",
    tz="UTC"
)

if len(expected_index) != len(df):
    raise ValueError(
        "Dataset does not contain a complete hourly timeline."
    )

print("Hourly timeline validated.")


# ============================================================
# 4. TEMPORAL FEATURES
# ============================================================

df["hour"] = df.index.hour
df["month"] = df.index.month
df["day_of_year"] = df.index.dayofyear


# ============================================================
# 5. CYCLICAL TEMPORAL FEATURES
# ============================================================

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

df["month_sin"] = np.sin(
    2 * np.pi * (df["month"] - 1) / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * (df["month"] - 1) / 12
)


# ============================================================
# 6. PRECIPITATION LAGS
# ============================================================

for lag in [1, 2, 3, 6]:
    df[f"precipitation_lag_{lag}h"] = (
        df["precipitation_mm"].shift(lag)
    )


# ============================================================
# 7. METEOROLOGICAL LAGS
# ============================================================

lag_variables = [
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",
    "wind_speed_ms",
]

for variable in lag_variables:
    df[f"{variable}_lag_1h"] = (
        df[variable].shift(1)
    )


# ============================================================
# 8. ROLLING PRECIPITATION FEATURES
# ============================================================

df["precipitation_sum_3h"] = (
    df["precipitation_mm"]
    .rolling(
        window=3,
        min_periods=3
    )
    .sum()
)

df["precipitation_sum_6h"] = (
    df["precipitation_mm"]
    .rolling(
        window=6,
        min_periods=6
    )
    .sum()
)


# ============================================================
# 9. SHORT-TERM METEOROLOGICAL CHANGES
# ============================================================

df["temperature_change_1h"] = (
    df["temperature_c"]
    - df["temperature_c"].shift(1)
)

df["humidity_change_1h"] = (
    df["humidity_pct"]
    - df["humidity_pct"].shift(1)
)

df["pressure_change_1h"] = (
    df["pressure_station_mb"]
    - df["pressure_station_mb"].shift(1)
)


# ============================================================
# 10. FUTURE PRECIPITATION TARGET CANDIDATES
# ============================================================

future_columns = []

for horizon in range(1, 7):
    column_name = f"future_precipitation_{horizon}h"

    df[column_name] = (
        df["precipitation_mm"]
        .shift(-horizon)
    )

    future_columns.append(column_name)


# ============================================================
# 11. DEFINE MODEL FEATURE COLUMNS
# ============================================================

feature_columns = [
    # Current meteorological conditions
    "precipitation_mm",
    "temperature_c",
    "humidity_pct",
    "pressure_station_mb",
    "wind_direction_deg",
    "wind_gust_ms",
    "wind_speed_ms",

    # Temporal features
    "hour_sin",
    "hour_cos",
    "month_sin",
    "month_cos",
    "day_of_year",

    # Precipitation history
    "precipitation_lag_1h",
    "precipitation_lag_2h",
    "precipitation_lag_3h",
    "precipitation_lag_6h",
    "precipitation_sum_3h",
    "precipitation_sum_6h",

    # Meteorological history
    "temperature_c_lag_1h",
    "humidity_pct_lag_1h",
    "pressure_station_mb_lag_1h",
    "wind_speed_ms_lag_1h",

    # Short-term changes
    "temperature_change_1h",
    "humidity_change_1h",
    "pressure_change_1h",
]


# ============================================================
# 12. IDENTIFY ROWS READY FOR MODELING
# ============================================================

df["features_complete"] = (
    df[feature_columns]
    .notna()
    .all(axis=1)
)

df["all_future_targets_available"] = (
    df[future_columns]
    .notna()
    .all(axis=1)
)

df["modeling_ready"] = (
    df["is_observed"]
    & df["features_complete"]
    & df["all_future_targets_available"]
)


# ============================================================
# 13. DATA LEAKAGE CHECK
# ============================================================

leaking_features = [
    column
    for column in feature_columns
    if column.startswith("future_")
]

if leaking_features:
    raise ValueError(
        f"Future information detected in feature set: "
        f"{leaking_features}"
    )


# ============================================================
# 14. SAVE FEATURE DATASET
# ============================================================

df.to_csv(
    OUTPUT_PATH,
    index=True
)


# ============================================================
# 15. FEATURE ENGINEERING SUMMARY
# ============================================================

print()
print("=== FEATURE ENGINEERING SUMMARY ===")
print()

print(f"Total rows: {len(df)}")
print(f"Model features created: {len(feature_columns)}")
print(f"Future horizons created: {len(future_columns)}")

print()
print(
    "Rows with complete model features:",
    int(df["features_complete"].sum())
)

print(
    "Rows with all +1h to +6h future values:",
    int(df["all_future_targets_available"].sum())
)

print(
    "Rows ready for future modeling:",
    int(df["modeling_ready"].sum())
)

print()
print("Future target candidates:")
for column in future_columns:
    print("-", column)

print()
print(f"Output saved to: {OUTPUT_PATH}")


display(
    df[
        feature_columns
        + future_columns
        + ["modeling_ready"]
    ].head(10)
)

Dataset loaded: (8760, 20)
Hourly timeline validated.

=== FEATURE ENGINEERING SUMMARY ===

Total rows: 8760
Model features created: 25
Future horizons created: 6

Rows with complete model features: 8159
Rows with all +1h to +6h future values: 8179
Rows ready for future modeling: 8039

Future target candidates:
- future_precipitation_1h
- future_precipitation_2h
- future_precipitation_3h
- future_precipitation_4h
- future_precipitation_5h
- future_precipitation_6h

Output saved to: ..\data\processed\recife_2018_features.csv


,precipitation_mm,temperature_c,humidity_pct,pressure_station_mb,wind_direction_deg,wind_gust_ms,wind_speed_ms,hour_sin,hour_cos,month_sin,...,temperature_change_1h,humidity_change_1h,pressure_change_1h,future_precipitation_1h,future_precipitation_2h,future_precipitation_3h,future_precipitation_4h,future_precipitation_5h,future_precipitation_6h,modeling_ready
datetime_utc,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000e+00,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 01:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.258819,9.659258e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 02:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.500000,8.660254e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 03:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.707107,7.071068e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 04:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,5.000000e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 05:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.965926,2.588190e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 06:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,6.123234e-17,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 07:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.965926,-2.588190e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
2018-01-01 08:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.866025,-5.000000e-01,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


## Conclusion

The feature engineering stage transformed the hourly meteorological observations into a structured set of temporal and meteorological predictors.

Lagged observations, rolling precipitation totals, cyclical temporal variables, and short-term atmospheric changes were created using only information available at or before each prediction timestamp.

Future precipitation values from +1h to +6h were also generated as target candidates and were explicitly excluded from the model feature set to prevent data leakage.

No final heavy-rainfall classification threshold is defined at this stage. The binary prediction targets will be created after evaluating the event distribution across the complete 2018–2024 dataset.